# Results overview

This notebook provides a quick overview of the current evaluation data in `results/eval_results.csv`.

It focuses on:

- run inventory and metadata,
- aggregate success/return metrics,
- comparison of experiment groups (for example OPD normalization variants).

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "visualization" else Path.cwd().resolve()
CSV_PATH = REPO_ROOT / "results" / "eval_results.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find {CSV_PATH}. Run notebook from repo root or visualization/.")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(f"Columns: {len(df.columns)}")

df.head(3)

In [ ]:
# Basic inventory
summary = {
    "rows": len(df),
    "unique_experiments": df["experiment_name"].nunique() if "experiment_name" in df else None,
    "unique_seeds": df["seed"].nunique() if "seed" in df else None,
    "global_steps": sorted(df["global_step"].dropna().unique().tolist()) if "global_step" in df else None,
    "configs": sorted(df["hydra_config_name"].dropna().unique().tolist()) if "hydra_config_name" in df else None,
}
summary

In [ ]:
# Core metric distributions
metrics = [m for m in ["success_at_end", "success_once", "return", "reward"] if m in df.columns]
if not metrics:
    raise ValueError("No expected metrics found in CSV")

fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4))
if len(metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics):
    sns.histplot(df[metric].dropna(), bins=15, kde=True, ax=ax)
    ax.set_title(metric)

plt.tight_layout()
plt.show()

In [ ]:
# Compare runs by experiment name (success_at_end)
if "experiment_name" in df.columns and "success_at_end" in df.columns:
    ordered = (
        df.groupby("experiment_name", as_index=False)["success_at_end"]
        .mean()
        .sort_values("success_at_end", ascending=False)
    )

    plt.figure(figsize=(max(8, 0.5 * len(ordered)), 4))
    sns.barplot(data=ordered, x="experiment_name", y="success_at_end", color="steelblue")
    plt.xticks(rotation=60, ha="right")
    plt.title("Mean success_at_end by experiment")
    plt.tight_layout()
    plt.show()

    ordered.head(10)
else:
    print("Required columns missing: experiment_name or success_at_end")